# 02 - LoRA fine-tune Llama-3.2-1B for cell-type annotation

This notebook fine-tunes a small open instruct model to classify RA synovial scRNA-seq cell clusters from their top marker genes, using LoRA. Run it on a Colab T4 GPU (Runtime > Change runtime type > T4 GPU).

I'm writing this assuming zero Hugging Face background, so a few concepts first.

**Base model vs. instruct model.** `meta-llama/Llama-3.2-1B-Instruct` is a 1-billion-parameter model Meta already trained on huge amounts of text, then further tuned to follow chat-style instructions (system/user/assistant turns) rather than just continuing arbitrary text. We start from that checkpoint. Nothing here trains from scratch.

**Fine-tuning, in this context.** We show the model many `(marker genes) -> (cell type label)` examples and adjust its weights so it gets better at producing the right label. This style, explicit input/output pairs, training the model to predict the output tokens, is called SFT (Supervised Fine-Tuning). It's the same "predict the next token" objective the base model already learned on, just pointed at our small labeled dataset now.

**LoRA, and why we don't fine-tune the whole model.** A 1B model has 1 billion weights. Updating all of them ("full fine-tuning") needs a lot of GPU memory for gradients and optimizer state on every weight, and risks the model forgetting general language ability given how little data we have (a few hundred examples). LoRA (Low-Rank Adaptation) freezes the original weights and injects small trainable adapter matrices next to a few layers, here the attention projections `q_proj`/`k_proj`/`v_proj`/`o_proj`. Those matrices have far fewer parameters than the layers they sit beside, so training touches a tiny fraction of the model, fits easily on a T4, and the saved adapter is a few MB instead of gigabytes. `r=16` below is the rank of those matrices: higher r means more trainable capacity and bigger files, lower means cheaper and more constrained. 16 is a common default for a task this size.

**Chat templates.** Instruct models expect input formatted as a specific sequence of special tokens marking who said what, not raw text. Every model ships its own template. We don't touch this manually below: we hand `SFTTrainer` a dataset in `{"prompt": [...], "completion": [...]}` format (a list of `{"role": ..., "content": ...}` turns) and it applies Llama 3.2's chat template internally. It also computes loss only on the completion (assistant) turn, ignoring the system/user prompt tokens. That's what we want: grade the model on predicting the label, not on reproducing the fixed instructions we wrote.

**One adapter per label level.** Levels 1/2/3 have different label sets (Immune/Stromal; T/NK/Myeloid/B/Plasma/Fibroblast; Treg/Tconv/CD8 T/Tph), so each gets its own LoRA adapter, trained and saved separately. The base model stays shared and untouched.

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate

## Setup: Drive and Hugging Face access

Two one-time things before running the cells below.

1. **Upload data.** Copy the 9 `data/examples_*.jsonl` files from this repo into Google Drive at `MyDrive/lora-scrnaseq-cell-type-annotation/data/`. Tiny files, about 250 lines total, drag and drop in the Drive web UI is fine.
2. **Hugging Face access.** `meta-llama/Llama-3.2-1B-Instruct` is gated. Meta requires accepting its license before you can download it.
   - Go to https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct and accept the license (instant, just needs a HF account).
   - Create an access token at https://huggingface.co/settings/tokens (read access is enough).
   - Add it as a Colab secret named `HF_TOKEN` (key icon, left sidebar), or run the login cell below and paste the token when prompted.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # falls back to an interactive token prompt

## Constants

`SYSTEM_PROMPT` and `USER_TEMPLATE` are copied verbatim from `build-plan.md`'s prompt format section. `03_eval.py` needs the exact same strings, otherwise baseline and fine-tuned aren't compared on the same prompt.

In [ ]:
import torch
from pathlib import Path

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

DRIVE_ROOT = Path("/content/drive/MyDrive/lora-scrnaseq-cell-type-annotation")
DATA_DIR = DRIVE_ROOT / "data"  # upload data/examples_*.jsonl here before running
ADAPTER_DIR = DRIVE_ROOT / "adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

LEVELS = [1, 2, 3]

# Shared prompt + label format (build-plan.md) -- 03_eval.py must reuse these verbatim.
SYSTEM_PROMPT = "You are a computational immunologist annotating RA synovial scRNA-seq clusters."
USER_TEMPLATE = (
    "Top marker genes for a cell cluster (RA synovium): {genes}.\n"
    "Classify into exactly one of: [{labels}].\n"
    "Answer with the label only."
)

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training hyperparameters
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3

## Data loading

Each line in `examples_{level}_{split}.jsonl` looks like `{"genes": ["FOXP3", "IL2RA", ...], "label": "Treg", "donor": "S022", "level": 3}` (from `01_dataprep.py`). We convert each into TRL's `prompt`/`completion` format from above. The label set in the prompt (`Classify into exactly one of: [...]`) is derived from the actual data at runtime instead of hardcoded, so it can't drift out of sync with the jsonl files.

In [ ]:
import json

from datasets import Dataset


def load_jsonl(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


def label_set_for_level(level: int) -> list[str]:
    labels = set()
    for split in ("train", "val", "test"):
        for example in load_jsonl(DATA_DIR / f"examples_{level}_{split}.jsonl"):
            labels.add(example["label"])
    return sorted(labels)


def to_conversational_dataset(examples: list[dict], label_set: list[str]) -> Dataset:
    labels_str = ", ".join(label_set)
    rows = []
    for example in examples:
        genes_str = ", ".join(example["genes"])
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(genes=genes_str, labels=labels_str)},
            ],
            "completion": [{"role": "assistant", "content": example["label"]}],
        })
    return Dataset.from_list(rows)

## Training function

`SFTConfig` bundles the usual `TrainingArguments` (batch size, learning rate, epochs, precision) with a few SFT-specific options.

- `fp16=True`: mixed precision training. `build-plan.md` said bf16, but bf16 needs an Ampere-or-newer GPU. Colab's free T4 is Turing-generation and only benefits from fp16. Same idea, smaller float format for memory and speed, different format for this GPU.
- `model_init_kwargs={"dtype": torch.float16}`: we pass `model=BASE_MODEL` as a string, so TRL loads it via `AutoModelForCausalLM.from_pretrained` internally. This sets the dtype for the frozen base weights.
- `eval_strategy="epoch"`: runs the val set through the model after every epoch, so eval loss shows up in the training log instead of just trusting training loss, which can look fine even while the model overfits the tiny train set.
- `peft_config=...`: passing this to `SFTTrainer` is what wraps the base model with trainable LoRA matrices. Without it this quietly becomes a full fine-tune of all 1B parameters.

Each call reloads a fresh copy of the frozen base model, since `model=BASE_MODEL` is a string, not a shared object. Each level's adapter trains independently from the same starting point.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def train_level(level: int) -> None:
    label_set = label_set_for_level(level)
    train_examples = load_jsonl(DATA_DIR / f"examples_{level}_train.jsonl")
    val_examples = load_jsonl(DATA_DIR / f"examples_{level}_val.jsonl")
    train_dataset = to_conversational_dataset(train_examples, label_set)
    val_dataset = to_conversational_dataset(val_examples, label_set)

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        task_type="CAUSAL_LM",
    )

    output_dir = ADAPTER_DIR / f"level{level}"
    training_args = SFTConfig(
        output_dir=str(output_dir),
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        fp16=True,
        eval_strategy="epoch",
        logging_steps=5,
        report_to="none",
        model_init_kwargs={"dtype": torch.float16},
    )

    trainer = SFTTrainer(
        model=BASE_MODEL,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        peft_config=peft_config,
    )

    print(f"--- Level {level}: {len(train_examples)} train / {len(val_examples)} val examples, labels={label_set} ---")
    trainer.train()
    trainer.save_model(str(output_dir))
    print(f"Saved adapter to {output_dir}")

## Run it

About 250 examples across 3 levels on a T4, expect well under 10 minutes total. Watch each level's per-epoch eval loss. If it climbs while training loss keeps dropping, that level (most likely Level 3, fewest examples) is overfitting. `fine-tune-plan.md`'s honest-scope note already expects this: one day of scope, not a hyperparameter-tuned model.

In [ ]:
for level in LEVELS:
    train_level(level)

## Spot check one adapter

Not a real evaluation. That's `03_eval.py`'s job: accuracy, macro-F1, confusion matrix on the full test sets, baseline vs. fine-tuned. This just confirms the saved adapter loads back onto the base model and produces a plausible label for one held-out Level 3 example, before you close this notebook.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

CHECK_LEVEL = 3

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR / f"level{CHECK_LEVEL}"))

label_set = label_set_for_level(CHECK_LEVEL)
test_example = load_jsonl(DATA_DIR / f"examples_{CHECK_LEVEL}_test.jsonl")[0]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_TEMPLATE.format(
        genes=", ".join(test_example["genes"]), labels=", ".join(label_set)
    )},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
output = model.generate(inputs, max_new_tokens=10)

print("Predicted:", tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip())
print("True label:", test_example["label"])